# Sinh prompt bang VLM — bien the VISION

Giong `Generate_Prompts_VLM.ipynb` nhung model duoc **nhin 3 anh binh thuong**
cua tung class thay vi chi doc ten class.

## Vi sao can bien the nay

P2b (blind) that bai: **31.16 `p_f1`**, thua ca ba muc con lai. Chan doan tu
`results/phase_a_ladder/README.md`:

| Class | P2b | Tot nhat kia | Chenh |
|---|---|---|---|
| `pill` | 11.10 | 63.94 | **-52.84** |
| `metal_nut` | 20.97 | 38.79 | -17.82 |
| `screw` | 5.86 | 20.81 | -14.95 |

Model doan defect tu **ten class** roi sinh `crack` / `stain` / `missing piece`
cho gan nhu moi thu. Defect `pill` trong MVTec la doi mau va in loi; defect
`screw` la ren hong o muc vi mo. Danh tu cu the SAI dan detector di sai cho -
te hon han mot chu `"defect."` von khong ma hoa gia dinh nao.

Bien the vision cho model **nhin** vat the truoc khi doan. Do la thu duy nhat
co the sua dung diem yeu da chan doan.

## Ranh gioi ro ri du lieu

`train_image_paths` khoa cung `train/good` va co test chan moi duong khac. MVTec
train split toan anh binh thuong nen hop le - model khong bao gio nhin thay
defect, chi nhin thay vat the trong khong co defect.

Xem spec muc 5.5. Luan van phai trich duoc dong code nay.

## Khac notebook blind o hai cho

- Can tai MVTec (~10 phut, can Kaggle credentials)
- Probe phai kiem shape khoi anh, khong chi kiem nap model

## Nguong

| | `p_f1` |
|---|---|
| P2b blind | 31.16 |
| P1 general | 35.80 |
| P3 thu cong | 37.44 |
| **Chon tot nhat moi class (oracle-4)** | **40.80** |

## Cai dat (~2 phut)

In [ ]:
%cd /content
!rm -rf /content/Segment-Any-Anomaly
!git clone -q -b dev https://github.com/SyDuc7421/Segment-Any-Anomaly.git
%cd Segment-Any-Anomaly

# Chi can transformers + accelerate. KHONG can bitsandbytes (chay fp16), khong
# can GroundingDINO/SAM (khong chay inference), khong can dataset (blind chi
# nhan ten class).
!pip install -q "transformers>=4.45" accelerate

import importlib.util
print('transformers:', 'OK' if importlib.util.find_spec('transformers') else 'THIEU')
print('torch       :', 'OK' if importlib.util.find_spec('torch') else 'THIEU')
!git log --oneline -1

## Tai MVTec (~10 phut, can Kaggle credentials)

Chi doc `train/good`. Khong cham anh test, ground-truth, hay anh anomaly.

In [ ]:
%cd /content/Segment-Any-Anomaly
%mkdir -p /content/datasets

from google.colab import userdata
import json, pathlib, os

pathlib.Path('/root/.kaggle').mkdir(exist_ok=True)
pathlib.Path('/root/.kaggle/kaggle.json').write_text(json.dumps({
    'username': userdata.get('KAGGLE_USERNAME'),
    'key': userdata.get('KAGGLE_KEY')
}))
!chmod 600 /root/.kaggle/kaggle.json

# Accept dataset terms first at: https://www.kaggle.com/datasets/ipythonx/mvtec-ad
!pip install -q kaggle
!kaggle datasets download -d ipythonx/mvtec-ad -p /content/datasets/ --unzip

os.environ['MVTEC_DIR'] = '/content/datasets'

# Nap danh sach class theo duong dan: `from datasets import ...` se an phai
# package datasets cua HuggingFace, va datasets/__init__.py cua repo lai keo theo
# loguru/cv2 ma notebook nhe nay khong cai.
import importlib.util

_s = importlib.util.spec_from_file_location(
    'repo_mvtec', '/content/Segment-Any-Anomaly/datasets/mvtec.py')
_m = importlib.util.module_from_spec(_s)
_s.loader.exec_module(_m)

present = [c for c in _m.mvtec_classes if os.path.isdir(f'/content/datasets/{c}')]
print(f'MVTec: {len(present)}/15 classes ready:', present)

# Chi train/good duoc doc - ranh gioi ro ri o spec muc 5.5.
import glob
n = sum(len(glob.glob(f'/content/datasets/{c}/train/good/*.png')) for c in present)
print(f'anh train/good: {n} (day la TOAN BO anh script duoc phep doc)')

## Probe: nap model + gui mot anh that (~3 phut)

In [ ]:
# Probe chay trong TIEN TRINH RIENG, khong phai trong kernel.
#
# Chay thang trong cell thi kernel giu model lai sau khi xong - `del` va
# `empty_cache()` khong tra het bo nho ve. Cell sinh prompt la mot tien trinh
# con, nen hai ban model cung nam tren GPU cung luc va T4 14.5 GB het cho:
#   "Process 1052 has 7.13 GiB in use ... this process has 7.43 GiB in use"
# Tien trinh rieng thoat la GPU sach.
%cd /content/Segment-Any-Anomaly
!python - <<'PY'
import glob
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
MAX_SIDE = 512

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map='auto'
).eval()
print('nap duoc:', type(model).__name__)

sample = sorted(glob.glob('/content/datasets/carpet/train/good/*.png'))[0]
img = Image.open(sample).convert('RGB')
print('anh thu :', sample, img.size, end=' ')
img.thumbnail((MAX_SIDE, MAX_SIDE), Image.LANCZOS)
print('-> ha xuong', img.size)

messages = [{'role': 'user', 'content': [
    {'type': 'image'},
    {'type': 'text', 'text': 'In one short sentence, what object is this?'},
]}]
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[img], return_tensors='pt').to(model.device)
out = model.generate(**inputs, max_new_tokens=48, do_sample=False)
print('tra loi :', processor.decode(out[0][inputs['input_ids'].shape[1]:],
                                    skip_special_tokens=True).strip())
print('VRAM    :', f'{torch.cuda.max_memory_allocated() / 1e9:.1f} GB')
PY

# Model tra loi dung "carpet"/"fabric"/"textile" nghia la no THAT SU nhin thay
# anh. Tra loi chung chung nghia la anh khong toi duoc model - dung chay tiep.

## Sinh prompt (~15 phut)

In [ ]:
%cd /content/Segment-Any-Anomaly
!python tools/gen_prompts.py \
    --dataset mvtec --variant vision \
    --model Qwen/Qwen2.5-VL-3B-Instruct --fp16 \
    --data-root /content/datasets --n-images 2 --max-image-side 512 \
    --out SAA/prompts/generated/mvtec-vision.json

## Doi chieu vision voi blind

In [ ]:
# Doi chieu vision voi blind: model nhin anh co doi cach dat prompt khong?
import importlib.util, json, os
from collections import Counter

spec = importlib.util.spec_from_file_location(
    'llm_prompts', '/content/Segment-Any-Anomaly/SAA/prompts/llm_prompts.py')
lp = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lp)

vision = lp.load_prompt_file('SAA/prompts/generated/mvtec-vision.json')
blind_path = 'SAA/prompts/generated/mvtec-blind.json'
blind = lp.load_prompt_file(blind_path) if os.path.exists(blind_path) else {}

print(f'{"class":12s} {"blind":48s} vision')
print('-' * 110)
for c in vision:
    b = ', '.join(p['text'] for p in blind[c]['defect_prompts']) if c in blind else '-'
    v = ', '.join(p['text'] for p in vision[c]['defect_prompts'])
    print(f'{c:12s} {b[:46]:48s} {v[:56]}')

n_v = sum(len(s['defect_prompts']) for s in vision.values())
uniq_v = len({p['text'] for s in vision.values() for p in s['defect_prompts']})
print(f'\nvision: {n_v/len(vision):.1f} prompt/class, {uniq_v} cum khac nhau tren {n_v}'
      f' -> lap lai {1-uniq_v/n_v:.0%}')
if blind:
    n_b = sum(len(s['defect_prompts']) for s in blind.values())
    uniq_b = len({p['text'] for s in blind.values() for p in s['defect_prompts']})
    print(f'blind : {n_b/len(blind):.1f} prompt/class, {uniq_b} cum khac nhau tren {n_b}'
          f' -> lap lai {1-uniq_b/n_b:.0%}')
    print('\nLap lai giam = model dua vao anh chu khong dua vao template.')

calls = 1 + n_v / len(vision)
print(f'\n{calls:.2f} luot DINO/anh; P3 thu cong dung 5.20')
print(f'Uoc t_total = {calls * 265 + 530:.0f} ms so voi 1926 ms cua P3 '
      f'-> {1926 / (calls * 265 + 530):.2f}x')

## Kiem nhiem du lieu huan luyen

In [ ]:
# Kiem nhiem du lieu huan luyen. Blind cho 0/53 trung khit - vision co the khac,
# vi nhin anh co the goi lai ky uc ve dataset nay.
import importlib.util

_s = importlib.util.spec_from_file_location(
    'mvtec_parameters',
    '/content/Segment-Any-Anomaly/SAA/prompts/mvtec_parameters.py')
_m = importlib.util.module_from_spec(_s)
_s.loader.exec_module(_m)
manual_prompts = _m.manual_prompts

exact = total_llm = 0
for cls, manual in manual_prompts.items():
    if cls not in vision:
        continue
    manual_texts = {p[0].strip().lower().rstrip('.') for p in manual}
    llm_texts = {p['text'].strip().lower().rstrip('.') for p in vision[cls]['defect_prompts']}
    overlap = manual_texts & llm_texts
    total_llm += len(llm_texts)
    exact += len(overlap)
    if overlap:
        print(f'{cls:12s} trung khit: {sorted(overlap)}')

print(f'\n{exact}/{total_llm} prompt trung khit tung chu voi prompt thu cong')
print('(blind cho 0/53)')

## Luu ra Drive

In [ ]:
from google.colab import drive
import os, shutil

try:
    drive.mount('/content/drive')
except ValueError:
    drive.mount('/content/drive', force_remount=True)

dst = '/content/drive/MyDrive/SAA_results/generated_prompts'
os.makedirs(dst, exist_ok=True)

for name in ('mvtec-vision.json', 'mvtec-vision-meta.json'):
    src = f'/content/Segment-Any-Anomaly/SAA/prompts/generated/{name}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'{os.path.getsize(src):>8,} B  {name}')

print(f'\n-> {dst}')
print('\nTai ve, dat vao SAA/prompts/generated/ roi commit.')
print("Sau do them 'P2v' vao LEVELS trong Phase_A_Prompt_Ladder.ipynb.")